# 02 - Classical Artificial Potential Field Navigation
This notebook demonstrates feature extraction, virtual attractive/repulsive force calculation via Artificial Potential Fields (APF), and full mission execution with the classical baseline navigator.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from utils.io_utils import load_json
from simulation.environment import UAVEnvironment
from classical.feature_extractor import NavigationFeatureExtractor
from classical.potential_field import ArtificialPotentialField
from classical.baseline import ClassicalBaselineNavigator
from navigation.controller import NavigationController
from dashboard.visualization import plot_trajectory_comparison


## 1. Feature Extraction from Environmental State


In [ ]:
map_data = load_json(PROJECT_ROOT / "data" / "maps" / "map_01.json")
scenario = {"start": [5.0, 5.0], "target": [45.0, 45.0], "max_steps": 100}

env = UAVEnvironment(map_data=map_data, scenario_data=scenario)
state = env.reset()

extractor = NavigationFeatureExtractor()
feat_vec, feat_dict = extractor.extract_features(state)
print("Extracted Feature Vector (Length = {}):".format(len(feat_vec)))
for k, v in feat_dict.items():
    print(f"  {k:<28}: {v:+.4f}")


## 2. Potential Field Force Analysis


In [ ]:
apf = ArtificialPotentialField()
f_total, breakdown = apf.compute_total_force(
    drone_pos=state["drone_position"],
    target_pos=state["target_position"],
    obstacles=env.obstacles,
)
print("Attractive Force Vector:", breakdown["f_att"])
print("Repulsive Force Vector (Obstacles):", breakdown["f_rep_obs"])
print("Combined APF Force Vector:", f_total)
print("Local Minima Flag:", breakdown["is_local_minimum"])


## 3. Running Classical Baseline Flight Mission


In [ ]:
navigator = ClassicalBaselineNavigator(obstacles=env.obstacles)
controller = NavigationController(environment=env, navigator=navigator)
controller.reset()

ep_result = controller.run_episode()
print("Mission Completed:", ep_result["success"])
print("Total Steps Taken:", ep_result["total_steps"])
print("Path Length Traveled:", ep_result["path_length"])
print("Obstacle Collisions:", ep_result["collision_count"])
print("Safety Interventions:", ep_result["safety_interventions"])

# Plot flight path
fig = plot_trajectory_comparison(
    scenario=scenario,
    map_data=map_data,
    classical_trajectory=ep_result["trajectory"],
    quantum_trajectory=[],
    title="Classical APF Trajectory",
)
plt.show()
